In [34]:
import json
import csv
import pandas as pd

In [35]:
input_file = "/Users/evansnow/Downloads/vulns-found-main-2026-01-15-thru-2026-04-15.ndjson"

In [36]:
with open(input_file) as file:
    rows = [json.loads(line) for line in file if line.strip()]
 
df = pd.DataFrame(rows)

In [86]:
# Clean up column names (lowercase, underscores)
df.columns = df.columns.str.strip().str.lower()#.str.replace(" ", "_")

#rename columns
df.rename(columns={'name': 'package', 'vulnerability': 'vuln_ID'}, inplace=True)
 
# Parse timestamps
df["startedat"] = pd.to_datetime(df["startedat"])
 
# Normalize categoricals
df["severity"] = pd.Categorical(df["severity"], categories=["Critical", "High", "Medium", "Low", "Negligible"], ordered=True)
df["type"] = df["type"].astype("category")
 
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   jobdatabaseid          557 non-null    int64              
 1   jobname                557 non-null    object             
 2   startedat              557 non-null    datetime64[ns, UTC]
 3   workflowdatabaseid     557 non-null    int64              
 4   workflowname           557 non-null    object             
 5   workflowrundatabaseid  557 non-null    int64              
 6   workflowrunname        557 non-null    object             
 7   package                557 non-null    object             
 8   installed              557 non-null    object             
 9   fixed in               557 non-null    object             
 10  type                   557 non-null    category           
 11  vuln_ID                557 non-null    object             

,jobdatabaseid,jobname,startedat,workflowdatabaseid,workflowname,workflowrundatabaseid,workflowrunname,package,installed,fixed in,type,vuln_ID,severity,epss,risk
0,69841146084,Vulnerability Scans (frontend) / Anchore Scan,2026-04-03 12:08:11+00:00,149604214,CI Weekly Vulnerability Report,23945647527,CI Weekly Vulnerability Report,lodash,4.17.23,4.18.0,npm,GHSA-r5fr-rjxr-66jc,High,< 0.1% (21st),< 0.1
1,69841146084,Vulnerability Scans (frontend) / Anchore Scan,2026-04-03 12:08:11+00:00,149604214,CI Weekly Vulnerability Report,23945647527,CI Weekly Vulnerability Report,next,16.1.6,16.1.7,npm,GHSA-ggv3-7p47-pfv8,Medium,< 0.1% (22nd),< 0.1
2,69841146084,Vulnerability Scans (frontend) / Anchore Scan,2026-04-03 12:08:11+00:00,149604214,CI Weekly Vulnerability Report,23945647527,CI Weekly Vulnerability Report,lodash,4.17.23,4.18.0,npm,GHSA-f23m-r3pf-42rh,Medium,< 0.1% (12th),< 0.1
3,69841146084,Vulnerability Scans (frontend) / Anchore Scan,2026-04-03 12:08:11+00:00,149604214,CI Weekly Vulnerability Report,23945647527,CI Weekly Vulnerability Report,next,16.1.6,16.1.7,npm,GHSA-3x4c-7xq6-9pq8,Medium,< 0.1% (4th),< 0.1
4,69841146084,Vulnerability Scans (frontend) / Anchore Scan,2026-04-03 12:08:11+00:00,149604214,CI Weekly Vulnerability Report,23945647527,CI Weekly Vulnerability Report,next,16.1.6,16.1.7,npm,GHSA-h27x-g6w4-24gq,Medium,< 0.1% (4th),< 0.1


In [87]:
len(df)

557

## Finding out how to identify "the same vulnerability"

In [88]:
# How many fully duplicated rows are there?
df.duplicated().sum()

np.int64(0)

### Check uniqueness of all column combinations with 'vulnerability'

In [89]:
cols = [c for c in df.columns if c != "vuln_ID"]
non_unique = []

for col in cols:
    is_unique = df[["vuln_ID", col]].duplicated().sum() == 0
    print(f"vuln_ID + {col}: {'unique' if is_unique else 'not unique'}")

vuln_ID + jobdatabaseid: not unique
vuln_ID + jobname: not unique
vuln_ID + startedat: not unique
vuln_ID + workflowdatabaseid: not unique
vuln_ID + workflowname: not unique
vuln_ID + workflowrundatabaseid: not unique
vuln_ID + workflowrunname: not unique
vuln_ID + package: not unique
vuln_ID + installed: not unique
vuln_ID + fixed in: not unique
vuln_ID + type: not unique
vuln_ID + severity: not unique
vuln_ID + epss: not unique
vuln_ID + risk: not unique


It looks like there are no true duplicate rows in the df, but no combination of 2 columns creates a unqiue identifier. This means the uniqueness is spread among 3 columns. Let's keep investigating...

In [33]:
from itertools import combinations

cols = df.columns.tolist()
for r in range(2, 5):
    for combo in combinations(cols, r):
        if df[list(combo)].duplicated().sum() == 0:
            print(f"Unique identifier found: {combo}")
            break
    else:
        continue
    break

Unique identifier found: ('jobdatabaseid', 'name', 'installed', 'vulnerability')


#### Check if fixed in version is unique

In [90]:
df.groupby(["vuln_ID", "package", "installed"])["fixed in"].nunique().max()

2

In [91]:
df.groupby(["vuln_ID", "package", "installed"])["fixed in"].nunique().reset_index(name="nunique").query("nunique > 1")

,vuln_ID,package,installed,nunique
35,CVE-2025-11468,python,3.14.2,2
39,CVE-2025-15282,python,3.14.2,2
40,CVE-2025-15366,python,3.14.2,2
42,CVE-2025-15367,python,3.14.2,2
44,CVE-2026-0672,python,3.14.2,2
45,CVE-2026-0865,python,3.14.2,2
46,CVE-2026-1299,python,3.14.2,2
72,GHSA-2g4f-4pwh-qvx6,ajv,6.12.6,2


In [92]:
inconsistent = df.groupby(["vuln_ID", "package", "installed"])["fixed in"].nunique().reset_index(name="nunique").query("nunique > 1")

df.merge(inconsistent[["vuln_ID", "package", "installed"]], on=["vuln_ID", "package", "installed"])[["vuln_ID", "package", "installed", "fixed in"]].drop_duplicates()

,vuln_ID,package,installed,fixed in
0,GHSA-2g4f-4pwh-qvx6,ajv,6.12.6,6.14.0
1,CVE-2026-0672,python,3.14.2,"3.13.12, *3.14.3, 3.15.0a6"
2,CVE-2026-0865,python,3.14.2,"3.13.12, *3.14.3, 3.15.0a6"
3,CVE-2025-15366,python,3.14.2,3.15.0a6
4,CVE-2025-15367,python,3.14.2,3.15.0a6
5,CVE-2025-15282,python,3.14.2,"3.13.12, *3.14.3, 3.15.0a6"
6,CVE-2026-1299,python,3.14.2,"3.13.12, *3.14.3, 3.15.0a6"
7,CVE-2025-11468,python,3.14.2,"3.13.12, *3.14.3, 3.15.0a6"
15,CVE-2026-0865,python,3.14.2,3.15.0
16,CVE-2025-15282,python,3.14.2,3.15.0


### Aggregate dates to find 'first seen' and 'fixed' dates per unique vulnerability

In [128]:
df2 = df.groupby(["vuln_ID", "package", "installed", "jobname"]).agg(
    first_seen=("startedat", "min"),
    fixed=("startedat", "max")
).reset_index()

df2["days_open"] = (df2["fixed"] - df2["first_seen"]).dt.days

In [129]:
df2.sample(10)

,vuln_ID,package,installed,jobname,first_seen,fixed,days_open
51,ALAS2023-2026-1514,kernel6.18-headers,1:6.18.8-9.213.amzn2023,Vulnerability Scans (api) / Anchore Scan,2026-03-28 12:07:35+00:00,2026-03-30 12:16:23+00:00,2
82,ALAS2023-2026-1568,rust-toolset-srpm-macros,1.93.0-1.amzn2023.0.1,Analytics Vulnerability Scans / Anchore Scan,2026-04-14 10:26:44+00:00,2026-04-14 10:26:44+00:00,0
85,ALAS2023-2026-1568,rust-toolset-srpm-macros,1.93.0-1.amzn2023.0.1,Vulnerability Scans / Anchore Scan,2026-04-14 14:58:25+00:00,2026-04-14 15:21:28+00:00,0
107,CVE-2025-11468,python,3.14.2,Vulnerability Scans (api) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
220,GHSA-mmwr-2jhp-mc7j,django,6.0.3,NOFOs Vulnerability Scans / Anchore Scan,2026-04-09 10:24:07+00:00,2026-04-14 10:26:44+00:00,5
88,ALAS2023-2026-1586,openssl,1:3.5.5-1.amzn2023.0.3,Vulnerability Scans (analytics) / Anchore Scan,2026-04-14 12:19:29+00:00,2026-04-14 14:40:40+00:00,0
174,CVE-2026-3479,python,3.14.3,Vulnerability Scans (api) / Anchore Scan,2026-04-10 12:11:57+00:00,2026-04-10 12:11:57+00:00,0
181,CVE-2026-4224,python,3.14.3,Vulnerability Scans (analytics) / Anchore Scan,2026-04-10 12:11:57+00:00,2026-04-10 12:11:57+00:00,0
207,GHSA-fvcv-3m26-pcqx,axios,1.13.6,Frontend Vulnerability Scans / Anchore Scan,2026-04-11 10:08:58+00:00,2026-04-13 10:31:26+00:00,2
215,GHSA-m959-cc7f-wv43,cryptography,46.0.5,API Vulnerability Scans / Anchore Scan,2026-03-28 10:07:15+00:00,2026-03-30 10:27:27+00:00,2


In [130]:
df2.sort_values(by='days_open', ascending=False)

,vuln_ID,package,installed,jobname,first_seen,fixed,days_open
115,CVE-2025-15282,python,3.14.2,Vulnerability Scans (analytics) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
116,CVE-2025-15282,python,3.14.2,Vulnerability Scans (api) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
106,CVE-2025-11468,python,3.14.2,Vulnerability Scans (analytics) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
128,CVE-2026-0865,python,3.14.2,Vulnerability Scans (api) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
127,CVE-2026-0865,python,3.14.2,Vulnerability Scans (analytics) / Anchore Scan,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
...,...,...,...,...,...,...,...
79,ALAS2023-2026-1542,libnghttp2,1.59.0-3.amzn2023.0.1,Vulnerability Scans (api) / Anchore Scan,2026-04-14 12:19:29+00:00,2026-04-14 14:40:40+00:00,0
80,ALAS2023-2026-1542,libnghttp2,1.59.0-3.amzn2023.0.1,Vulnerability Scans / Anchore Scan,2026-04-14 14:58:25+00:00,2026-04-14 15:21:28+00:00,0
81,ALAS2023-2026-1568,rust-toolset-srpm-macros,1.93.0-1.amzn2023.0.1,API Vulnerability Scans / Anchore Scan,2026-04-14 10:26:44+00:00,2026-04-14 10:26:44+00:00,0
82,ALAS2023-2026-1568,rust-toolset-srpm-macros,1.93.0-1.amzn2023.0.1,Analytics Vulnerability Scans / Anchore Scan,2026-04-14 10:26:44+00:00,2026-04-14 10:26:44+00:00,0


In [131]:
df2.value_counts('days_open')

days_open
0     156
1      24
2      15
56     14
14      7
5       6
21      6
3       5
6       2
20      2
41      2
Name: count, dtype: int64

In [133]:
#Output to CSV, keep commented out until ready to output

#df2.to_csv('vuln_scans.csv', index=False)

#### TEST

In [121]:
df_test = df2[df2['vuln_ID'] == 'CVE-2025-15366']

df_test
#df_test.sort_values(by = 'fixed')

,vuln_ID,package,installed,first_seen,fixed,days_open
40,CVE-2025-15366,python,3.14.2,2026-01-23 12:02:36+00:00,2026-03-20 12:05:26+00:00,56
41,CVE-2025-15366,python,3.14.3,2026-03-27 12:10:17+00:00,2026-04-10 12:11:57+00:00,14


### Vulnerabilities by Count of Appearances

In [26]:
df_vuln = df['vulnerability'].value_counts().reset_index()

df_vuln = df_vuln.merge(
   df[["vulnerability", "severity"]].drop_duplicates(),
    on="vulnerability"
)
df_vuln = df_vuln.merge(
   df[["vulnerability", "risk"]].drop_duplicates(),
    on="vulnerability"
)


df_vuln.head(15)

,vulnerability,count,severity,risk
0,ALAS2023-2026-1586,36,High,< 0.1
1,ALAS2023-2026-1522,24,Low,< 0.1
2,CVE-2025-15366,24,Medium,< 0.1
3,CVE-2025-15367,24,Medium,< 0.1
4,CVE-2025-12781,18,Medium,< 0.1
5,CVE-2025-15282,18,Medium,< 0.1
6,CVE-2026-0865,18,Medium,< 0.1
7,CVE-2026-0672,18,Medium,< 0.1
8,CVE-2025-11468,18,Medium,< 0.1
9,GHSA-m959-cc7f-wv43,15,Low,N/A
